# Sphere Packing Analysis (DEM-GPU)

This notebook demonstrates using the **DEM-GPU** engine to generate and analyze a sphere packing. We will:
1. Initialize the simulation with Spheres (Shape Type 0).
2. Run the physics engine to settle the particles.
3. Visualize the final configuration.
4. Compute the Radial Distribution Function (RDF) to characterize the structure.

In [ ]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Add build directory to path to find the compiled module
sys.path.append(os.path.abspath('../build'))
import demgpu

## 1. Simulation Setup

We initialize a simulation with 1000 particles using the **Sphere** shape type (0). We then expand them slightly (scale=1.0) and let them settle under gravity.

In [ ]:
NUM_PARTICLES = 1000
sim = demgpu.Simulation(NUM_PARTICLES)
sim.initialize(shape_type=0) # 0 = Sphere

# Set scales to 1.0 (Radius = 0.5 * 1.0 = 0.5)
scales = np.ones(NUM_PARTICLES, dtype=np.float32)
sim.set_scales(scales)

print("Running simulation...")
t0 = time.time()
# Run for 200 steps (enough for initial settling)
for _ in range(200):
    sim.step(0.002)
    
print(f"Simulation finished in {time.time() - t0:.3f}s")

## 2. Extract Data

We retrieve the final positions from the GPU.

In [ ]:
pos_data = sim.get_positions().reshape(-1, 4)
positions = pos_data[:, :3] # X, Y, Z

print(f"Position Range: X[{positions[:,0].min():.2f}, {positions[:,0].max():.2f}]")
print(f"Position Range: Y[{positions[:,1].min():.2f}, {positions[:,1].max():.2f}]")
print(f"Position Range: Z[{positions[:,2].min():.2f}, {positions[:,2].max():.2f}]")

## 3. Visualization

We can plot a subset of particles or a slice to verify stacking.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot a random subset for clarity
subset = positions # Plot all 1000
sc = ax.scatter(subset[:,0], subset[:,2], subset[:,1], c=subset[:,1], cmap='viridis', s=20)

# Adjust view to see the pile
ax.set_xlabel('X')
ax.set_ylabel('Z') # Swap Y/Z visually if needed, but here Z is depth
ax.set_zlabel('Y (Height)')
plt.colorbar(sc, label='Height')
plt.title("Final Particle Positions")
plt.show()

## 4. Radial Distribution Function (RDF)

We compute $g(r)$ to look for structure (peaks at 2R, etc).

In [ ]:
def compute_rdf(positions, r_max, dr, rho):
    n = len(positions)
    bins = np.arange(0, r_max + dr, dr)
    distances = []
    
    # Brute force (fine for N=1000)
    for i in range(n):
        for j in range(i + 1, n):
            d = np.linalg.norm(positions[i] - positions[j])
            if d <= r_max:
                distances.append(d)
                
    h, _ = np.histogram(distances, bins=bins)
    
    radii = (bins[:-1] + bins[1:]) / 2
    shell_vols = 4 * np.pi * (radii**2) * dr
    
    # Expected in shell: rho * V
    # Pairs scaling: N * (rho * V) / 2
    expected = (n * shell_vols * rho) / 2.0
    
    # Handle division by zero for small radii
    with np.errstate(divide='ignore', invalid='ignore'):
        g_r = h / expected
        g_r[np.isnan(g_r)] = 0
        
    return radii, g_r

# Estimate density roughly within the bounding box of the pile
min_p = positions.min(axis=0)
max_p = positions.max(axis=0)
vol_box = np.prod(max_p - min_p)
rho = NUM_PARTICLES / vol_box

radii, g_r = compute_rdf(positions, r_max=4.0, dr=0.1, rho=rho)

plt.figure(figsize=(8, 5))
plt.plot(radii, g_r, '-o')
plt.axvline(1.0, color='r', linestyle='--', label='Diameter (2R)')
plt.xlabel("Distance r")
plt.ylabel("g(r)")
plt.title("Radial Distribution Function")
plt.legend()
plt.grid(True)
plt.show()